In [ ]:
import glob
from music21 import converter, instrument, note, chord

def parse_midi_files(midi_folder: str) -> list:
    notes = []
    midi_files = glob.glob(midi_folder + "/*.mid")

    for file in midi_files:
        midi = converter.parse(file)

        parts = instrument.partitionByInstrument(midi)
        if parts:
            notes_to_parse = parts.parts[0].recurse()
        else:
            notes_to_parse = midi.flat.notes

        for element in notes_to_parse:
            if isinstance(element, note.Note):
                notes.append(str(element.pitch))
            elif isinstance(element, chord.Chord):
                notes.append('.'.join(str(n) for n in element.normalOrder))

    return notes


In [ ]:
# Vocabulary mapping
unique_notes = sorted(set(notes))
vocab_size = len(unique_notes)

note_to_int = {note: i for i, note in enumerate(unique_notes)}
int_to_note = {i: note for i, note in enumerate(unique_notes)}


In [ ]:
# Sequence creation
import numpy as np

sequence_length = 50
X = []
y = []

for i in range(len(notes) - sequence_length):
    seq_in = notes[i:i + sequence_length]
    seq_out = notes[i + sequence_length]

    X.append([note_to_int[n] for n in seq_in])
    y.append(note_to_int[seq_out])

X = np.array(X)
y = np.array(y)


In [ ]:
# Model
from tensorflow import keras
from tensorflow.keras import layers

model = keras.Sequential([
    layers.Embedding(input_dim=vocab_size, output_dim=128, input_length=sequence_length),
    layers.SimpleRNN(256, return_sequences=True),
    layers.SimpleRNN(256),
    layers.Dense(vocab_size)
])

model.summary()


In [ ]:
# Compile
loss_fn = keras.losses.SparseCategoricalCrossentropy(from_logits=True)

model.compile(
    optimizer='adam',
    loss=loss_fn,
    metrics=['accuracy']
)


In [ ]:
# Callback
reduce_lr_cb = keras.callbacks.ReduceLROnPlateau(
    monitor='loss',
    factor=0.5,
    patience=5,
    verbose=1
)


In [ ]:
# Temperature Sampling
import numpy as np

def sample_with_temperature(predictions, temperature=1.0):
    predictions = predictions / temperature
    exp_preds = np.exp(predictions)
    probs = exp_preds / np.sum(exp_preds)
    return np.random.choice(len(probs), p=probs)


##  Theory Answers

### MIDI vs Audio
- MIDI is symbolic (notes, pitch, duration)
- Audio is waveform (continuous)
- MIDI is easier for RNN

### Sequence Length
- Number of sequences = N - T
- Larger T = more context but slower

### RNN Parameters
Params = H(H + D + 1)

### Temperature
- Low → repetitive
- Medium → balanced
- High → random
